# Things still may go wrong while sending the request in streamiong and structured output
1. cut off - stop_reason -> length
2. Refusal - check for message.refusal in response
3. wrong facts - shaoe is fins but extra information is given
4. Network or server problems -

In [3]:

import time

from openai import (
    OpenAI,
    LengthFinishReasonError,
    APIError,
)
from pydantic import BaseModel, Field


# 1. Create the local Ollama client
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    max_retries=0,  # We handle retries ourselves
)


# 2. Define the expected invoice structure
class Invoice(BaseModel):
    invoice_number: str
    total_amount: float
    is_paid: bool


# 3. Human review function
def send_to_human(text, reason):
    print(f"Sent to human review. Reason: {reason}")
    print(f"Original input: {text}")


# 4. Extract invoice information
def extract_invoice(text, max_tries=3):

    for attempt in range(1, max_tries + 1):

        try:
            # Structured output using Pydantic
            completion = client.chat.completions.parse(
                model="qwen2.5:3b",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Extract invoice details from the user's text. "
                            "Return the invoice number, total amount as a "
                            "number, and whether the invoice is paid. "
                            "If payment status is unclear, do not assume "
                            "that the invoice is paid."
                        ),
                    },
                    {
                        "role": "user",
                        "content": text,
                    },
                ],
                response_format=Invoice,
                max_completion_tokens=200,
            )

            # Fetch the message, not just its content
            message = completion.choices[0].message

            # Check whether the model refused
            if message.refusal:
                send_to_human(text, "model_refused")
                return None

            # Fetch the parsed Pydantic object
            invoice = message.parsed

            if invoice is None:
                send_to_human(text, "structured_output_missing")
                return None

            # Validate business rules
            if invoice.total_amount <= 0:
                send_to_human(
                    text,
                    "amount is zero or negative",
                )
                return None

            return invoice

        except LengthFinishReasonError:
            send_to_human(text, "reply was cut off")
            return None

        except APIError as error:
            print(
                f"Attempt no. {attempt} failed: {error}. "
                "Retrying if attempts remain..."
            )

            if attempt < max_tries:
                # Exponential backoff: 2, 4, 8, ...
                time.sleep(2 ** attempt)

    send_to_human(text, "API keeps failing")
    return None


# 5. Test the function
user_message = (
    "Bill INV-2050, total Rs 12,000, not yet paid."
)

invoice = extract_invoice(user_message)

print(invoice)

if invoice is not None:
    print(invoice.model_dump())


invoice_number='INV-2050' total_amount=12000.0 is_paid=False
{'invoice_number': 'INV-2050', 'total_amount': 12000.0, 'is_paid': False}


# API -

1. API-key - mandatory -
2. model, message, max_tokens
3. usage, stop_reason
4. API calls fails -
  1. RateLimitError -> sending the request quite fast
  2. Server Busy or overloaded - 500
  3. Connection problem -
  4. Bad key -

5. API calls are stateless - having no state between the calls
6. Streaming and strutured output

#

openai - system and user messages both goes into message arg,
whereas in anthropic - system message goes under system parameter and user and assistant message goes under messages -
gemini - system = model , user_messages = input



In [ ]:

import time
from openai import OpenAI, APIError


# --------------------------------------------------
# 1. Ollama local client
# --------------------------------------------------

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    max_retries=0
)


# --------------------------------------------------
# 2. Human review function
# --------------------------------------------------

def send_to_human(text, reason):
    print(f"Sent to human review. Reason: {reason}")
    print(f"Original input: {text}")


# --------------------------------------------------
# 3. Unified local AI function
# --------------------------------------------------

def ask_ai(system, user, max_tokens=100, max_tries=3):

    for attempt in range(1, max_tries + 1):

        try:
            response = ollama_client.chat.completions.create(
                model="qwen2.5:3b",
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user}
                ],
                max_tokens=max_tokens
            )

            message = response.choices[0].message
            text = message.content or ""

            cutoff_reason = (
                response.choices[0].finish_reason == "length"
            )

            total_tokens = (
                response.usage.total_tokens
                if response.usage is not None
                else None
            )

            if cutoff_reason:
                send_to_human(user, "response_cut_off")
                return {
                    "output_text": text,
                    "total_tokens": total_tokens,
                    "cutoff_reason": True
                }

            return {
                "output_text": text,
                "total_tokens": total_tokens,
                "cutoff_reason": False
            }

        except APIError as error:
            print(
                f"Attempt {attempt} failed: {error}. "
                "Retrying if attempts remain..."
            )

            if attempt < max_tries:
                time.sleep(2 ** attempt)

        except Exception as error:
            send_to_human(user, f"unexpected_error: {error}")
            return None

    send_to_human(user, "API keeps failing")
    return None


# --------------------------------------------------
# 4. Test the local model
# --------------------------------------------------

if __name__ == "__main__":

    result = ask_ai(
        system="You explain shipment delays simply.",
        user="Container #88 stuck at Mumbai port, customs hold.",
        max_tokens=250
    )

    if result is not None:
        print("\nGenerated response:")
        print(result["output_text"])

        print("\nTotal tokens:")
        print(result["total_tokens"])

        print("\nResponse cut off:")
        print(result["cutoff_reason"])


Sent to human review. Reason: response_cut_off
Original input: Container #88 stuck at Mumbai port, customs hold.

Generated response:
Let's break down the situation of Container #88 being stuck at the Mumbai port due to the customs hold:

1. **Container #88**: This refers to a specific shipping container that was loaded with goods for export.
   
2. **Mumbai Port**: The container is being held at this port in India. Ports are points where large ships transfer cargo onto smaller vessels or onto land for delivery to inland destinations.

3. **Customs Hold**: This means that the goods inside Container #

Total tokens:
131

Response cut off:
True
